# SAFE rollouts：个人 Drive 副本 → 服务器（V3）

公共文件已经触发 Google Drive 下载配额，`gdown` 无法继续。运行本 Notebook 前：

1. 登录 Google Drive，打开 [Pi0-FAST 文件](https://drive.google.com/file/d/13z_cdwnaJota2iHkZbhYgVALujZwtM3b/view) 或 [OpenVLA/WidowX 文件](https://drive.google.com/file/d/1EwaccasZjnlM9L6SEYyWqTd7d6-BR9zp/view)。
2. 选择“制作副本”，把副本放入 `我的云端硬盘/SAFE-rollouts`。
3. 将副本重命名为原始文件名：`pi0fast_droid_0510_all.zip` 或 `openvla_widowx.zip`。

可以一次只复制一份。上传并验证后删除个人副本、清空回收站，再处理第二份。文件经 Colab 直接传入服务器，不经过本地电脑。

In [ ]:
!apt-get -qq update && apt-get -qq install -y sshpass rsync >/dev/null
!pip -q install 'paramiko>=3.5.0'
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

SOURCE_DIR = Path('/content/drive/MyDrive/SAFE-rollouts')
FILENAMES = [
    'pi0fast_droid_0510_all.zip',
    'openvla_widowx.zip',
]
available = []
for filename in FILENAMES:
    path = SOURCE_DIR / filename
    if path.is_file():
        available.append(path)
        print(f'已找到：{path} ({path.stat().st_size:,} bytes)')
    else:
        print(f'未找到，暂时跳过：{path}')
if not available:
    raise FileNotFoundError(
        '请先在个人 Drive 创建 SAFE-rollouts 文件夹，并放入至少一份已重命名的 ZIP 副本。'
    )

In [ ]:
import base64
import getpass
import hashlib
import os
import shlex
import socket
import stat
import subprocess

import paramiko

HOST = 'connect.bjb2.seetacloud.com'
PORT = 20559
USER = 'root'
REMOTE_DIR = '/root/autodl-tmp/datasets/safe-rollouts'
EXPECTED_HOST_KEY = 'SHA256:liZ36vNCsNcNdXeWs4f+g5ZIhPM/ZihP834vxs8Ulqc'
HOST_PUBLIC_KEY = 'ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIK3kwiFvUVgx1wVKa/LqYm6JNHZqht3d+OSpVJVG/AWa'
KNOWN_HOSTS = Path('/content/safe_known_hosts')
KNOWN_HOSTS.write_text(
    f'[{HOST}]:{PORT} {HOST_PUBLIC_KEY}\n', encoding='utf-8'
)
password = getpass.getpass('服务器密码（输入不会显示）：')

def open_sftp():
    sock = socket.create_connection((HOST, PORT), timeout=30)
    transport = paramiko.Transport(sock)
    transport.start_client(timeout=30)
    server_key = transport.get_remote_server_key()
    fingerprint = 'SHA256:' + base64.b64encode(
        hashlib.sha256(server_key.asbytes()).digest()
    ).decode().rstrip('=')
    if fingerprint != EXPECTED_HOST_KEY:
        transport.close()
        raise RuntimeError(f'服务器指纹不匹配：{fingerprint}')
    transport.auth_password(USER, password)
    return transport, paramiko.SFTPClient.from_transport(transport)

def mkdir_p(sftp, path):
    current = ''
    for part in path.strip('/').split('/'):
        current += '/' + part
        try:
            mode = sftp.stat(current).st_mode
            if not stat.S_ISDIR(mode):
                raise RuntimeError(f'远程路径不是目录：{current}')
        except FileNotFoundError:
            sftp.mkdir(current)

def remote_final_size(filename):
    transport, sftp = open_sftp()
    try:
        mkdir_p(sftp, REMOTE_DIR)
        try:
            return sftp.stat(f'{REMOTE_DIR}/{filename}').st_size
        except FileNotFoundError:
            return None
    finally:
        sftp.close()
        transport.close()

ssh_command = (
    f'ssh -p {PORT} -o StrictHostKeyChecking=yes '
    f'-o UserKnownHostsFile={shlex.quote(str(KNOWN_HOSTS))} '
    '-o Compression=no -o ServerAliveInterval=30 -o ServerAliveCountMax=10 '
    '-c aes128-gcm@openssh.com'
)
child_env = os.environ.copy()
child_env['SSHPASS'] = password

for source_path in available:
    filename = source_path.name
    local_size = source_path.stat().st_size
    existing_size = remote_final_size(filename)
    if existing_size is not None:
        if existing_size != local_size:
            raise RuntimeError(
                f'服务器已有同名文件但大小不同：{filename}'
            )
        print(f'服务器已有完整同尺寸文件，跳过：{filename}')
        continue

    remote_part = f'{REMOTE_DIR}/{filename}.part'
    print(f'Drive 副本 → 服务器（rsync 断点续传）：{filename}')
    subprocess.run(
        [
            'sshpass', '-e', 'rsync', '-ah', '--no-compress',
            '--partial', '--append-verify', '--info=progress2',
            '-e', ssh_command, str(source_path),
            f'{USER}@{HOST}:{remote_part}',
        ],
        check=True, env=child_env,
    )

    transport, sftp = open_sftp()
    try:
        uploaded_size = sftp.stat(remote_part).st_size
        if uploaded_size != local_size:
            raise RuntimeError(
                f'大小不一致：Drive {local_size:,}，服务器 {uploaded_size:,}'
            )
        sftp.rename(remote_part, f'{REMOTE_DIR}/{filename}')
    finally:
        sftp.close()
        transport.close()
    print(f'完成：{filename} ({local_size:,} bytes)')

password = None
child_env.pop('SSHPASS', None)
print('当前放入个人 Drive 的 SAFE 文件已传输完成。')